# Experiment 61 - Event Power Fine Tune

**Building on:** exp60 (OOF cmAP=0.95742), current best OOF.

**Hypothesis:** exp60 selected the lowest event power tested (0.9), while texture power stayed at 1.2. Fine-tune lower event power and nearby weights with texture power fixed around its optimum.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp60_path = nb_dir / 'exp60_taxon_dual_power.ipynb'
exp60_nb = json.loads(exp60_path.read_text())

# Reuse exp60 setup and dual-power helper definitions by executing its first two code cells.
for cell_idx in [1, 2]:
    print(f'--- executing exp60 cell {cell_idx} ---')
    src = ''.join(exp60_nb['cells'][cell_idx]['source'])
    if cell_idx == 2:
        # Only keep helper definitions and constants before the exp60 search starts.
        src = src.split("best_cmap = -1.0")[0]
    exec(compile(src, f'exp60_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp61: lower event power fine tune

TTA_W0S = [0.85, 0.90, 0.95]
POWER_EVENTS = [0.40, 0.60, 0.75, 0.90, 1.00]
POWER_TEXTURES = [1.1, 1.2, 1.3]
ALPHA_TEXTURES = [0.25, 0.30, 0.35]
ENSEMBLE_GRIDS = [
    (0.02, 0.56, 0.42),
    (0.02, 0.54, 0.44),
    (0.02, 0.58, 0.40),
    (0.00, 0.58, 0.42),
]

best_cmap = -1.0
best_cfg = {}
results = []

for wp, wm, wpr in ENSEMBLE_GRIDS:
    fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
    fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
    for tta_w0 in TTA_W0S:
        fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
        for power_event in POWER_EVENTS:
            for power_texture in POWER_TEXTURES:
                for alpha_texture in ALPHA_TEXTURES:
                    probs = postprocess_taxon_dual_power(
                        fp_avg, power_event=power_event, power_texture=power_texture,
                        alpha_event=0.10, alpha_texture=alpha_texture)
                    cmap = padded_cmap(Y_FULL_aligned, probs)
                    auc = macro_auc(Y_FULL_aligned, probs)
                    row = {'tta_w0': tta_w0, 'power_event': power_event, 'power_texture': power_texture,
                           'alpha_texture': alpha_texture, 'wp': wp, 'wm': wm, 'wpr': wpr,
                           'auc': auc, 'cmap': cmap}
                    results.append(row)
                    if cmap > best_cmap:
                        best_cmap = cmap
                        best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 30 configs:')
print(df.head(30).to_string(index=False))
print('\nBest cmAP by power_event:')
print(df.groupby('power_event')['cmap'].max().reset_index().to_string(index=False))

fp_0_f = best_cfg['wp'] * oof_proto_0 + best_cfg['wm'] * oof_mlp_0 + best_cfg['wpr'] * prior_0
fp_250_f = best_cfg['wp'] * oof_proto_250 + best_cfg['wm'] * oof_mlp_250 + best_cfg['wpr'] * prior_250
fp_f = best_cfg['tta_w0'] * fp_0_f + (1.0 - best_cfg['tta_w0']) * fp_250_f
p_best = postprocess_taxon_dual_power(fp_f, power_event=best_cfg['power_event'],
                                      power_texture=best_cfg['power_texture'],
                                      alpha_event=0.10,
                                      alpha_texture=best_cfg['alpha_texture'])
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)
cmap_tex = padded_cmap(Y_FULL_aligned[:, TEXTURE_MASK], p_best[:, TEXTURE_MASK])
cmap_evt = padded_cmap(Y_FULL_aligned[:, ~TEXTURE_MASK], p_best[:, ~TEXTURE_MASK])

print('=' * 60)
print('Experiment 61 - Event power fine tune')
print(f"Best: tta_w0={best_cfg['tta_w0']}, power_event={best_cfg['power_event']}, power_texture={best_cfg['power_texture']}, alpha_texture={best_cfg['alpha_texture']}")
print(f"Weights: wp={best_cfg['wp']}, wm={best_cfg['wm']}, wpr={best_cfg['wpr']}")
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'Texture cmAP={cmap_tex:.5f}  Event cmAP={cmap_evt:.5f}')
print(f'vs exp60 OOF (0.95742): delta cmAP = {cmap_b - 0.95742:+.5f}')
print(f'vs exp46 OOF (0.95579): delta cmAP = {cmap_b - 0.95579:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
